# Field Scan : test du modèle « Agroforestry inventory » (banane)

Modèle cité par le sujet : https://huggingface.co/JefersonPMS/agroforestry-inventory
(code : https://github.com/COURAGEOUS-LAND/agroforestry-inventory, GPL-3.0 ; poids CC BY 4.0).
Détecteur RF-DETR, un modèle par espèce, entraîné sur une seule ferme du Brésil (133 bananiers, images à 1,66 cm par pixel).

On le teste **tel quel** (sans réentraînement) sur la ferme de validation **Manuel Macias**, ramenée à 1,66 cm par pixel,
avec la même règle que pour les autres méthodes : une détection est juste si elle est à moins de 1 m d'un vrai bananier,
à l'intérieur de la parcelle. Ce modèle n'a jamais vu ReforesTree : pas de fuite.

Données attendues sur le Drive, dans `MyDrive/field_scan/test_agroforestry/` (produites par `outils/preparer_test_agroforestry.py`) :
`Manuel_Macias_1p66cm.tif`, `verite_Manuel_Macias.csv`, `parcelle_Manuel_Macias.geojson`.

Un GPU est conseillé (Exécution → Modifier le type d'exécution → GPU T4), mais pas obligatoire.

In [ ]:
# 1. Installation du modèle (code des auteurs) et accès au Drive
!git clone -q https://github.com/COURAGEOUS-LAND/agroforestry-inventory /content/agroforestry-inventory
%cd /content/agroforestry-inventory
!pip install -q -r requirements.txt
!python download_models.py
from google.colab import drive
drive.mount('/content/drive')
DOSSIER = '/content/drive/MyDrive/field_scan/test_agroforestry'
!ls "{DOSSIER}"

In [ ]:
# 2. Détection des bananiers sur Manuel Macias, avec la commande des auteurs
#    Confiance minimale très basse (0,05) : on garde tout, et on choisit le meilleur seuil à l'étape 3.
#    La fusion des détections proches suit le réglage recommandé par les auteurs pour la banane.
#    Après la détection, leur script ouvre une carte web qui tourne sans fin : « timeout » l'arrête
#    au bout de 20 minutes (la détection prend environ 2 minutes sur GPU), les résultats sont déjà écrits.
!timeout 1200 python inventory.py "{DOSSIER}/Manuel_Macias_1p66cm.tif" --species banana --confidence 0.05 --no-browser
!ls outputs

In [ ]:
# 3. Notation à 1 m, à l'intérieur de la parcelle, pour plusieurs seuils de confiance
import glob
import numpy as np, pandas as pd, geopandas as gpd
from scipy.optimize import linear_sum_assignment
from shapely import contains_xy

METRIQUE = 'EPSG:32717'   # UTM en mètres
fichier = glob.glob('outputs/*banana*centroids.geojson')[0]
detections = gpd.read_file(fichier).to_crs(METRIQUE)
zone = gpd.read_file(f'{DOSSIER}/parcelle_Manuel_Macias.geojson').to_crs(METRIQUE).geometry.union_all()
verite = pd.read_csv(f'{DOSSIER}/verite_Manuel_Macias.csv')
verite = verite[verite.est_banane]
vrais = gpd.GeoSeries(gpd.points_from_xy(verite.lon, verite.lat), crs=4326).to_crs(METRIQUE)
vrais = np.c_[vrais.x, vrais.y]
vrais = vrais[contains_xy(zone, vrais[:, 0], vrais[:, 1])]
det = np.c_[detections.geometry.x, detections.geometry.y]
dedans = contains_xy(zone, det[:, 0], det[:, 1])
det, confiance = det[dedans], detections.confidence.to_numpy()[dedans]
print(f'{len(vrais)} vrais bananiers dans la parcelle, {len(det)} détections dans la parcelle')

def noter(seuil, rayon=1.0):
    d = det[confiance >= seuil]
    if len(d) == 0:
        return {'seuil': seuil, 'detections': 0, 'justes': 0, 'precision': 0, 'rappel': 0, 'f1': 0}
    distances = np.linalg.norm(d[:, None] - vrais[None], axis=2)
    cout = np.where(distances <= rayon, distances, 1e6)
    l, c = linear_sum_assignment(cout)
    vp = int((cout[l, c] <= rayon).sum())
    p, r = vp / len(d), vp / len(vrais)
    return {'seuil': seuil, 'detections': len(d), 'justes': vp, 'precision': round(p, 3), 'rappel': round(r, 3),
            'f1': round(2 * p * r / (p + r), 3) if vp else 0.0,
            'erreur_comptage': round((len(d) - len(vrais)) / len(vrais), 3)}

tableau = pd.DataFrame([noter(s) for s in np.round(np.arange(0.05, 0.96, 0.05), 2)])
print(tableau.to_string(index=False))
meilleur = tableau.loc[tableau.f1.idxmax()]
print()
print(f"Seuil par défaut des auteurs (0,25) : F1 = {tableau.loc[tableau.seuil == 0.25, 'f1'].iloc[0]:.3f}")
print(f'Meilleur seuil sur la validation ({meilleur.seuil}) : F1 = {meilleur.f1:.3f}')
print()
print('Rappel, sur Manuel Macias : méthode simple F1 = 0,205 ; DeepForest avant apprentissage 0,208 ; après apprentissage 0,291')

In [ ]:
# 4. Sauvegarde sur le Drive
from pathlib import Path
sortie = Path('/content/drive/MyDrive/field_scan/resultats_comptage')
sortie.mkdir(exist_ok=True)
tableau.to_csv(sortie / 'agroforestry_inventory_manuel.csv', index=False)
detections.to_crs(4326).to_file(sortie / 'agroforestry_inventory_manuel_detections.geojson', driver='GeoJSON')
print('Écrit dans', sortie)